In [13]:
import numpy as np
import json
import joblib
import os

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)
from xgboost import XGBClassifier

os.makedirs("../models", exist_ok=True)

diseases = ["diabetes", "heart", "liver", "kidney"]
all_metrics = {}

for disease in diseases:
    print(f"\n{'='*50}")
    print(f"Training: {disease.upper()}")
    print(f"{'='*50}")

    X_train = np.load(f"../datasets/processed/{disease}_X_train.npy")
    X_test  = np.load(f"../datasets/processed/{disease}_X_test.npy")
    y_train = np.load(f"../datasets/processed/{disease}_y_train.npy")
    y_test  = np.load(f"../datasets/processed/{disease}_y_test.npy")

    print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

    all_metrics[disease] = {}

    models_to_train = [
        ("lr",  LogisticRegression(max_iter=1000, random_state=42)),
        ("rf",  RandomForestClassifier(
                    n_estimators=100,
                    random_state=42,
                    n_jobs=-1,
                    class_weight='balanced'
                )),
        ("xgb", XGBClassifier(
                    n_estimators=100,
                    random_state=42,
                    eval_metric="logloss",
                    verbosity=0,
                    scale_pos_weight=10
                )),
    ]

    for mtype, base_model in models_to_train:
        print(f"\n  [{mtype.upper()}] Training {disease}...")

        # Confidence Calibration using Platt Scaling
        model = CalibratedClassifierCV(
            base_model,
            method="sigmoid",
            cv=5
        )
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        acc  = round(accuracy_score(y_test, y_pred), 4)
        prec = round(precision_score(y_test, y_pred, zero_division=0), 4)
        rec  = round(recall_score(y_test, y_pred, zero_division=0), 4)
        f1   = round(f1_score(y_test, y_pred, zero_division=0), 4)
        auc  = round(roc_auc_score(y_test, y_prob), 4)

        print(f"    Accuracy:  {acc}")
        print(f"    Precision: {prec}")
        print(f"    Recall:    {rec}")
        print(f"    F1 Score:  {f1}")
        print(f"    AUC-ROC:   {auc}")

        # Save model
        path = f"../models/{disease}_{mtype}.pkl"
        joblib.dump(model, path)
        print(f"    Saved: {path}")

        all_metrics[disease][mtype] = {
            "model_type": {
                "lr":  "Logistic Regression",
                "rf":  "Random Forest",
                "xgb": "XGBoost"
            }[mtype],
            "accuracy":   acc,
            "precision":  prec,
            "recall":     rec,
            "f1_score":   f1,
            "auc_roc":    auc,
            "calibrated": True,
            "calibration_method": "Platt Scaling"
        }

# Save all metrics
with open("../models/all_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=4)

print("\n" + "="*50)
print("ALL 12 MODELS TRAINED AND SAVED")
print("="*50)

# Print final summary
print(f"\n{'Disease':<12} {'Model':<22} {'Accuracy':<10} {'AUC-ROC'}")
print("-" * 55)
for disease, models in all_metrics.items():
    for mtype, m in models.items():
        print(f"{disease:<12} {m['model_type']:<22} {m['accuracy']:<10} {m['auc_roc']}")


Training: DIABETES
Train size: (56554, 21), Test size: (14139, 21)

  [LR] Training diabetes...
    Accuracy:  0.746
    Precision: 0.7373
    Recall:    0.7642
    F1 Score:  0.7505
    AUC-ROC:   0.8232
    Saved: ../models/diabetes_lr.pkl

  [RF] Training diabetes...
    Accuracy:  0.7382
    Precision: 0.7233
    Recall:    0.7717
    F1 Score:  0.7467
    AUC-ROC:   0.8171
    Saved: ../models/diabetes_rf.pkl

  [XGB] Training diabetes...
    Accuracy:  0.7365
    Precision: 0.6818
    Recall:    0.8871
    F1 Score:  0.771
    AUC-ROC:   0.8257
    Saved: ../models/diabetes_xgb.pkl

Training: HEART
Train size: (353366, 25), Test size: (48494, 25)

  [LR] Training heart...
    Accuracy:  0.7564
    Precision: 0.2374
    Recall:    0.7832
    F1 Score:  0.3643
    AUC-ROC:   0.8438
    Saved: ../models/heart_lr.pkl

  [RF] Training heart...
    Accuracy:  0.9058
    Precision: 0.3806
    Recall:    0.09
    F1 Score:  0.1456
    AUC-ROC:   0.8131
    Saved: ../models/heart_rf.pkl


In [14]:
# Re-train heart models only with class weight fix
import numpy as np
import json
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)
from xgboost import XGBClassifier
import pandas as pd

disease = "heart"

# Load SMOTE-balanced arrays for training
X_train = np.load(f"../datasets/processed/{disease}_X_train.npy")
X_test  = np.load(f"../datasets/processed/{disease}_X_test.npy")
y_train = np.load(f"../datasets/processed/{disease}_y_train.npy")
y_test  = np.load(f"../datasets/processed/{disease}_y_test.npy")

# Get REAL class ratio from original clean CSV — NOT from SMOTE arrays
df_orig = pd.read_csv(f"../datasets/processed/{disease}_clean.csv")
neg = (df_orig["target"] == 0).sum()
pos = (df_orig["target"] == 1).sum()
scale = round(neg / pos, 2)
print(f"Real scale_pos_weight from original data: {scale}")
# This should be around 10-11 for heart disease

models_to_retrain = [
    ("rf", RandomForestClassifier(
               n_estimators=100,
               random_state=42,
               n_jobs=-1,
               class_weight='balanced'
           )),
    ("xgb", XGBClassifier(
               n_estimators=100,
               random_state=42,
               eval_metric="logloss",
               verbosity=0,
               scale_pos_weight=scale   # ← now uses real ratio ~10
           )),
]

with open("../models/all_metrics.json", "r") as f:
    all_metrics = json.load(f)

for mtype, base_model in models_to_retrain:
    print(f"\n  Retraining heart {mtype}...")

    model = CalibratedClassifierCV(base_model, method="sigmoid", cv=5)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc  = round(accuracy_score(y_test, y_pred), 4)
    prec = round(precision_score(y_test, y_pred, zero_division=0), 4)
    rec  = round(recall_score(y_test, y_pred, zero_division=0), 4)
    f1   = round(f1_score(y_test, y_pred, zero_division=0), 4)
    auc  = round(roc_auc_score(y_test, y_prob), 4)

    print(f"    Accuracy:  {acc}")
    print(f"    Precision: {prec}")
    print(f"    Recall:    {rec}")
    print(f"    F1:        {f1}")
    print(f"    AUC-ROC:   {auc}")

    joblib.dump(model, f"../models/{disease}_{mtype}.pkl")

    all_metrics[disease][mtype] = {
        "model_type": {"rf": "Random Forest", "xgb": "XGBoost"}[mtype],
        "accuracy":   acc,
        "precision":  prec,
        "recall":     rec,
        "f1_score":   f1,
        "auc_roc":    auc,
        "calibrated": True,
        "calibration_method": "Platt Scaling"
    }
    print(f"    Saved: ../models/{disease}_{mtype}.pkl")

with open("../models/all_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=4)

print("\nHeart models retrained and saved.")

Real scale_pos_weight from original data: 10.22

  Retraining heart rf...
    Accuracy:  0.9058
    Precision: 0.3806
    Recall:    0.09
    F1:        0.1456
    AUC-ROC:   0.8131
    Saved: ../models/heart_rf.pkl

  Retraining heart xgb...
    Accuracy:  0.9035
    Precision: 0.4239
    Recall:    0.23
    F1:        0.2982
    AUC-ROC:   0.7837
    Saved: ../models/heart_xgb.pkl

Heart models retrained and saved.


In [16]:
import numpy as np
import json
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("=" * 50)
print("RETRAINING HEART — NO SMOTE, CLASS WEIGHT ONLY")
print("=" * 50)

# ── Load original clean CSV — NOT the SMOTE arrays ──
df_heart = pd.read_csv("../datasets/processed/heart_clean.csv")

X = df_heart.drop("target", axis=1)
y = df_heart["target"]

# ── Scale ────────────────────────────────────────────
scaler = joblib.load("../models/heart_scaler.pkl")
X_scaled = scaler.transform(X)

# ── Split — NO SMOTE ─────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")
print(f"Train positive rate: {y_train.mean():.3f}")
print(f"Test positive rate:  {y_test.mean():.3f}")

# ── Calculate scale_pos_weight for XGB ───────────────
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = round(neg / pos, 2)
print(f"scale_pos_weight: {scale}")

# ── Models to retrain ─────────────────────────────────
models_to_retrain = [
    ("rf", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'
    )),
    ("xgb", XGBClassifier(
        n_estimators=100,
        random_state=42,
        eval_metric="logloss",
        verbosity=0,
        scale_pos_weight=scale
    )),
]

with open("../models/all_metrics.json", "r") as f:
    all_metrics = json.load(f)

for mtype, base_model in models_to_retrain:
    print(f"\n  Retraining heart {mtype} (no SMOTE)...")

    # ── NO Platt Scaling wrapper — model handles
    #    calibration internally via class_weight ───────
    base_model.fit(X_train, y_train)

    y_pred = base_model.predict(X_test)
    y_prob = base_model.predict_proba(X_test)[:, 1]

    acc  = round(accuracy_score(y_test, y_pred), 4)
    prec = round(precision_score(y_test, y_pred, zero_division=0), 4)
    rec  = round(recall_score(y_test, y_pred, zero_division=0), 4)
    f1   = round(f1_score(y_test, y_pred, zero_division=0), 4)
    auc  = round(roc_auc_score(y_test, y_prob), 4)

    print(f"    Accuracy:  {acc}")
    print(f"    Precision: {prec}")
    print(f"    Recall:    {rec}")
    print(f"    F1:        {f1}")
    print(f"    AUC-ROC:   {auc}")

    joblib.dump(base_model, f"../models/heart_{mtype}.pkl")

    all_metrics["heart"][mtype] = {
        "model_type": {"rf": "Random Forest", "xgb": "XGBoost"}[mtype],
        "accuracy":   acc,
        "precision":  prec,
        "recall":     rec,
        "f1_score":   f1,
        "auc_roc":    auc,
        "calibrated": False,
        "note": "No SMOTE — class_weight/scale_pos_weight used instead"
    }
    print(f"    Saved: ../models/heart_{mtype}.pkl")

with open("../models/all_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=4)

print("\nHeart RF and XGB retrained without SMOTE.")

RETRAINING HEART — NO SMOTE, CLASS WEIGHT ONLY
Train size: (193972, 25), Test size: (48494, 25)
Train positive rate: 0.089
Test positive rate:  0.089
scale_pos_weight: 10.22

  Retraining heart rf (no SMOTE)...
    Accuracy:  0.9087
    Precision: 0.4144
    Recall:    0.0599
    F1:        0.1047
    AUC-ROC:   0.8136
    Saved: ../models/heart_rf.pkl

  Retraining heart xgb (no SMOTE)...
    Accuracy:  0.7578
    Precision: 0.2355
    Recall:    0.7647
    F1:        0.3601
    AUC-ROC:   0.8344
    Saved: ../models/heart_xgb.pkl

Heart RF and XGB retrained without SMOTE.


In [17]:
import numpy as np
import json
import joblib
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)

print("Fixing Heart RF via threshold adjustment...")

# ── Load the already-saved RF model ──────────────────
rf_model = joblib.load("../models/heart_rf.pkl")

# ── Load test data ────────────────────────────────────
X_test  = np.load("../datasets/processed/heart_X_test.npy")
y_test  = np.load("../datasets/processed/heart_y_test.npy")

# NOTE: heart_X_test.npy is from the original SMOTE pipeline
# We need the no-SMOTE test set — reload from CSV
import pandas as pd
from sklearn.model_selection import train_test_split

df_heart = pd.read_csv("../datasets/processed/heart_clean.csv")
X = df_heart.drop("target", axis=1)
y = df_heart["target"]

scaler = joblib.load("../models/heart_scaler.pkl")
X_scaled = scaler.transform(X)

_, X_test_real, _, y_test_real = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ── Get probability scores ────────────────────────────
y_prob = rf_model.predict_proba(X_test_real)[:, 1]

# ── Find best threshold by F1 score ──────────────────
print("\nThreshold search:")
print(f"{'Threshold':>10} {'Accuracy':>10} {'Precision':>10} "
      f"{'Recall':>8} {'F1':>8}")
print("-" * 55)

best_f1 = 0
best_threshold = 0.5
best_metrics = {}

for threshold in np.arange(0.05, 0.50, 0.025):
    y_pred_t = (y_prob >= threshold).astype(int)

    acc  = accuracy_score(y_test_real, y_pred_t)
    prec = precision_score(y_test_real, y_pred_t, zero_division=0)
    rec  = recall_score(y_test_real, y_pred_t, zero_division=0)
    f1   = f1_score(y_test_real, y_pred_t, zero_division=0)

    print(f"{threshold:>10.3f} {acc:>10.4f} {prec:>10.4f} "
          f"{rec:>8.4f} {f1:>8.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold
        best_metrics = {
            "accuracy": round(acc, 4),
            "precision": round(prec, 4),
            "recall": round(rec, 4),
            "f1_score": round(f1, 4),
            "auc_roc": round(roc_auc_score(y_test_real, y_prob), 4),
            "threshold": round(threshold, 3)
        }

print(f"\nBest threshold: {best_threshold:.3f}")
print(f"Best metrics:   {best_metrics}")

# ── Save threshold alongside model ────────────────────
joblib.dump(best_threshold, "../models/heart_rf_threshold.pkl")
print(f"\nThreshold saved: ../models/heart_rf_threshold.pkl")

# ── Update all_metrics.json ───────────────────────────
with open("../models/all_metrics.json", "r") as f:
    all_metrics = json.load(f)

all_metrics["heart"]["rf"] = {
    "model_type": "Random Forest",
    **best_metrics,
    "calibrated": False,
    "note": f"Threshold adjusted to {best_threshold:.3f} for optimal F1"
}

with open("../models/all_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=4)

print("all_metrics.json updated.")

Fixing Heart RF via threshold adjustment...

Threshold search:
 Threshold   Accuracy  Precision   Recall       F1
-------------------------------------------------------
     0.050     0.6330     0.1775   0.8579   0.2941
     0.075     0.7065     0.2044   0.7927   0.3250
     0.100     0.7573     0.2279   0.7214   0.3463
     0.125     0.7835     0.2426   0.6738   0.3568
     0.150     0.8142     0.2643   0.6081   0.3685
     0.175     0.8314     0.2788   0.5625   0.3729
     0.200     0.8527     0.3032   0.5025   0.3782
     0.225     0.8630     0.3152   0.4579   0.3734
     0.250     0.8751     0.3297   0.3889   0.3569
     0.275     0.8816     0.3399   0.3482   0.3440
     0.300     0.8892     0.3505   0.2848   0.3143
     0.325     0.8945     0.3669   0.2538   0.3001
     0.350     0.8991     0.3787   0.2059   0.2668
     0.375     0.9014     0.3853   0.1791   0.2445
     0.400     0.9038     0.3899   0.1398   0.2058
     0.425     0.9055     0.3989   0.1196   0.1841
     0.450    

In [18]:
import numpy as np
import json
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)
from xgboost import XGBClassifier

print("=" * 50)
print("RETRAINING LIVER — STRONGER REGULARIZATION")
print("=" * 50)

# ── Load SMOTE arrays (keep SMOTE for liver) ─────────
X_train = np.load("../datasets/processed/liver_X_train.npy")
X_test  = np.load("../datasets/processed/liver_X_test.npy")
y_train = np.load("../datasets/processed/liver_y_train.npy")
y_test  = np.load("../datasets/processed/liver_y_test.npy")

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

# ── Stronger regularization ───────────────────────────
models_to_retrain = [
    ("rf", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        max_depth=6,            # ← reduced from 10
        min_samples_leaf=20,    # ← increased from 10
        min_samples_split=30,   # ← added
        max_features="sqrt"     # ← limit features per split
    )),
    ("xgb", XGBClassifier(
        n_estimators=100,
        random_state=42,
        eval_metric="logloss",
        verbosity=0,
        max_depth=3,            # ← reduced from 5
        learning_rate=0.05,     # ← slow learning
        subsample=0.7,          # ← use 70% of rows per tree
        colsample_bytree=0.7,   # ← use 70% of features per tree
        reg_alpha=1.0,          # ← stronger L1
        reg_lambda=5.0,         # ← stronger L2
        min_child_weight=10     # ← prevents splits on tiny groups
    )),
]

with open("../models/all_metrics.json", "r") as f:
    all_metrics = json.load(f)

for mtype, base_model in models_to_retrain:
    print(f"\n  Retraining liver {mtype}...")

    base_model.fit(X_train, y_train)

    y_pred = base_model.predict(X_test)
    y_prob = base_model.predict_proba(X_test)[:, 1]

    acc  = round(accuracy_score(y_test, y_pred), 4)
    prec = round(precision_score(y_test, y_pred, zero_division=0), 4)
    rec  = round(recall_score(y_test, y_pred, zero_division=0), 4)
    f1   = round(f1_score(y_test, y_pred, zero_division=0), 4)
    auc  = round(roc_auc_score(y_test, y_prob), 4)

    print(f"    Accuracy:  {acc}")
    print(f"    Precision: {prec}")
    print(f"    Recall:    {rec}")
    print(f"    F1:        {f1}")
    print(f"    AUC-ROC:   {auc}")

    joblib.dump(base_model, f"../models/liver_{mtype}.pkl")

    all_metrics["liver"][mtype] = {
        "model_type": {"rf": "Random Forest", "xgb": "XGBoost"}[mtype],
        "accuracy":   acc,
        "precision":  prec,
        "recall":     rec,
        "f1_score":   f1,
        "auc_roc":    auc,
        "calibrated": False,
        "note": "Regularized to prevent overfitting"
    }
    print(f"    Saved: ../models/liver_{mtype}.pkl")

with open("../models/all_metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=4)

print("\nLiver models retrained with stronger regularization.")

RETRAINING LIVER — STRONGER REGULARIZATION
Train size: (35066, 10), Test size: (6139, 10)

  Retraining liver rf...
    Accuracy:  0.7698
    Precision: 0.9937
    Recall:    0.682
    F1:        0.8089
    AUC-ROC:   0.9371
    Saved: ../models/liver_rf.pkl

  Retraining liver xgb...
    Accuracy:  0.726
    Precision: 0.9524
    Recall:    0.6487
    F1:        0.7718
    AUC-ROC:   0.881
    Saved: ../models/liver_xgb.pkl

Liver models retrained with stronger regularization.


In [19]:
import pandas as pd
import numpy as np

df = pd.read_csv("../datasets/processed/liver_clean.csv")

print("Correlation with target:")
corr = df.corr()["target"].abs().sort_values(ascending=False)
print(corr)

Correlation with target:
target                                  1.000000
Direct Bilirubin                        0.247275
Total Bilirubin                         0.222905
Alkphos Alkaline Phosphotase            0.180797
Sgpt Alamine Aminotransferase           0.164709
ALB Albumin                             0.159076
Sgot Aspartate Aminotransferase         0.157425
A/G Ratio Albumin and Globulin Ratio    0.156430
Total Protiens                          0.030134
Age of the patient                      0.004761
Gender                                  0.003190
Name: target, dtype: float64
